In [21]:
import joblib
import numpy as np
import os

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

In [22]:
save_dir = "data/processed"

raw_data    = np.load(os.path.join(save_dir, "raw_data.npy"))
temperature = np.load(os.path.join(save_dir, "temperature.npy"))
train_idx   = np.load(os.path.join(save_dir, "train_idx.npy"))
val_idx     = np.load(os.path.join(save_dir, "val_idx.npy"))
test_idx    = np.load(os.path.join(save_dir, "test_idx.npy"))
scaler      = joblib.load(os.path.join(save_dir, "scaler.pkl"))

num_train_samples = len(train_idx)
num_val_samples   = len(val_idx)
num_test_samples  = len(test_idx)

temp_mean = scaler.mean_[1]
temp_std  = scaler.scale_[1]

print(f"raw_data:    {raw_data.shape}")
print(f"temperature: {temperature.shape}")
print(f"train/val/test: {num_train_samples} / {num_val_samples} / {num_test_samples}")

temperature_normalized = (temperature - temp_mean) / temp_std

raw_data:    (420451, 14)
temperature: (420451,)
train/val/test: 210225 / 105112 / 105114


In [23]:
sampling_rate   = 6
sequence_length = 120
delay           = sampling_rate * (sequence_length + 24 - 1)
batch_size      = 256

class TimeseriesDataset(Dataset):
    def __init__(self, data, targets, sequence_length, sampling_rate, 
                 start_index, end_index, shuffle=False):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate

        # Valid starting indices: each sequence of length sequence_length
        # sampled every sampling_rate steps needs:
        # (sequence_length - 1) * sampling_rate + 1 rows ahead
        self.indices = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Sample every `sampling_rate` steps for `sequence_length` steps
        steps = np.arange(start, start + self.sequence_length * self.sampling_rate, 
                          self.sampling_rate)
        x = self.data[steps]
        # Target is `delay` steps ahead of the sequence start
        y = self.targets[start + delay]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)


In [24]:
train_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0, end_index=num_train_samples,
)

val_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples, end_index=num_train_samples + num_val_samples,
)

test_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples + num_val_samples, end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

# Quick sanity check
for inputs, targets in train_loader:
    print("Input shape:", inputs.shape)   # (batch_size, sequence_length, num_features)
    print("Target shape:", targets.shape) # (batch_size,)
    break

Input shape: torch.Size([256, 120, 14])
Target shape: torch.Size([256])


In [25]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss = 0.0
    total_mae  = 0.0
    n          = 0
    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds = model(inputs)
            loss  = criterion(preds, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * inputs.size(0)
            total_mae  += torch.sum(torch.abs(preds - targets)).item()
            n          += inputs.size(0)
   # mae_celsius = (total_mae / n) * temp_std
    return total_loss / n, total_mae / n   # MAE already in °C


def get_predictions(model, dataset):
    model.eval()
    all_preds     = []
    all_targets   = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)
    with torch.no_grad():
        for inputs, targets in loader:
            preds = model(inputs.to(device)).cpu().numpy()
            all_preds.append(preds * temp_std + temp_mean)  # un-normalize predictions
            all_targets.append(targets.numpy())              # targets already in °C
    return np.concatenate(all_preds), np.concatenate(all_targets)

In [36]:
class ModelCheckpoint:
    """Saves the best model based on a monitored metric."""
    def __init__(self, filepath, monitor="val_mae", mode="min", verbose=True):
        self.filepath = filepath
        self.monitor  = monitor
        self.verbose  = verbose
        self.best     = float("inf") if mode == "min" else float("-inf")
        self.mode     = mode

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = value < self.best if self.mode == "min" else value > self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Best model saved ({self.monitor}: {value:.2f}°C)")
        return improved


class EarlyStopping:
    """Stops training when a monitored metric stops improving."""
    def __init__(self, monitor="val_mae", patience=5, min_delta=1e-4, mode="min"):
        self.monitor     = monitor
        self.patience    = patience
        self.min_delta   = min_delta
        self.mode        = mode
        self.best        = float("inf") if mode == "min" else float("-inf")
        self.counter     = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best    = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  Early stopping triggered (no improvement for {self.patience} epochs)")
        return improved


class ReduceLROnPlateau:
    """Wraps PyTorch scheduler with the same callback interface."""
    def __init__(self, optimizer, monitor="val_mae", patience=3, factor=0.5):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

In [27]:
class SimpleRNNModel(nn.Module):
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.rnn  = nn.RNN(input_size=num_features, hidden_size=hidden_size,
                           batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # out: (batch, seq_len, hidden_size)
        # return_sequences=False equivalent → take last timestep
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)

In [28]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = SimpleRNNModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_simple_rnn")

callbacks = [
    ModelCheckpoint("jena_simple_rnn_best.pt", monitor="val_mae"),
]

# --- Training loop ---
epochs = 10
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

writer.close()

Epoch 01 — train loss: 56.5205, train MAE: 5.58°C | val loss: 23.2507, val MAE: 3.51°C
  ✓ Best model saved (val_mae: 3.51°C)
Epoch 02 — train loss: 16.8565, train MAE: 3.08°C | val loss: 12.4502, val MAE: 2.64°C
  ✓ Best model saved (val_mae: 2.64°C)
Epoch 03 — train loss: 12.1998, train MAE: 2.69°C | val loss: 10.3438, val MAE: 2.46°C
  ✓ Best model saved (val_mae: 2.46°C)
Epoch 04 — train loss: 11.0461, train MAE: 2.58°C | val loss: 9.6011, val MAE: 2.39°C
  ✓ Best model saved (val_mae: 2.39°C)
Epoch 05 — train loss: 10.5642, train MAE: 2.53°C | val loss: 9.3253, val MAE: 2.37°C
  ✓ Best model saved (val_mae: 2.37°C)
Epoch 06 — train loss: 10.3243, train MAE: 2.50°C | val loss: 9.1326, val MAE: 2.35°C
  ✓ Best model saved (val_mae: 2.35°C)
Epoch 07 — train loss: 10.1809, train MAE: 2.49°C | val loss: 9.1406, val MAE: 2.35°C
Epoch 08 — train loss: 10.0740, train MAE: 2.47°C | val loss: 8.9860, val MAE: 2.33°C
  ✓ Best model saved (val_mae: 2.33°C)
Epoch 09 — train loss: 9.9641, train

In [29]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("jena_simple_rnn_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")


Test MAE: 2.50°C


In [30]:
class StackedRNNModel(nn.Module):
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.rnn = nn.RNN(input_size=num_features, hidden_size=hidden_size,
                  num_layers=2, batch_first=True)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x, _   = self.rnn(x)    
        return self.head(x[:, -1, :]).squeeze(-1)

In [31]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = StackedRNNModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_stacked_rnn")

callbacks = [
    ModelCheckpoint("jena_stacked_rnn_best.pt", monitor="val_mae"),
]

# --- Training loop ---
epochs = 10
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model) if isinstance(cb, ModelCheckpoint) else cb.step(metrics)

writer.close()

Epoch 01 — train loss: 55.0968, train MAE: 5.47°C | val loss: 23.4461, val MAE: 3.51°C
  ✓ Best model saved (val_mae: 3.51°C)
Epoch 02 — train loss: 16.7285, train MAE: 3.05°C | val loss: 12.7221, val MAE: 2.66°C
  ✓ Best model saved (val_mae: 2.66°C)
Epoch 03 — train loss: 11.9962, train MAE: 2.66°C | val loss: 10.4539, val MAE: 2.48°C
  ✓ Best model saved (val_mae: 2.48°C)
Epoch 04 — train loss: 10.8150, train MAE: 2.55°C | val loss: 9.5087, val MAE: 2.38°C
  ✓ Best model saved (val_mae: 2.38°C)
Epoch 05 — train loss: 10.2768, train MAE: 2.50°C | val loss: 9.3620, val MAE: 2.37°C
  ✓ Best model saved (val_mae: 2.37°C)
Epoch 06 — train loss: 9.9456, train MAE: 2.46°C | val loss: 8.9830, val MAE: 2.33°C
  ✓ Best model saved (val_mae: 2.33°C)
Epoch 07 — train loss: 9.6895, train MAE: 2.42°C | val loss: 8.9545, val MAE: 2.33°C
  ✓ Best model saved (val_mae: 2.33°C)
Epoch 08 — train loss: 9.4889, train MAE: 2.40°C | val loss: 8.5615, val MAE: 2.27°C
  ✓ Best model saved (val_mae: 2.27°C)


In [33]:
# --- Reload best and evaluate ---
model.load_state_dict(torch.load("jena_stacked_rnn_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")


Test MAE: 2.48°C


In [34]:
class BidirectionalRNNModel(nn.Module):
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.rnn  = nn.LSTM(input_size=num_features, hidden_size=hidden_size,
                           batch_first=True, bidirectional=True)
        # bidirectional doubles the output size
        self.head = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)

In [37]:
# --- Setup ---
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = BidirectionalRNNModel(num_features=raw_data.shape[-1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.MSELoss()
writer    = SummaryWriter(log_dir="runs/jena_bidirectional_rnn")

callbacks = [
    ModelCheckpoint("jena_bidirectional_rnn_best.pt", monitor="val_mae"),
    EarlyStopping(monitor="val_mae", patience=7),
    ReduceLROnPlateau(optimizer, monitor="val_mae", patience=3, factor=0.5),
]

# --- Training loop ---
epochs = 50
for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model, val_loader,   criterion)

    metrics = {"train_loss": train_loss, "train_mae": train_mae,
               "val_loss":   val_loss,   "val_mae":   val_mae}

    writer.add_scalars("Loss", {"train": train_loss, "val": val_loss}, epoch)
    writer.add_scalars("MAE",  {"train": train_mae,  "val": val_mae},  epoch)

    print(f"Epoch {epoch:02d} — "
          f"train loss: {train_loss:.4f}, train MAE: {train_mae:.2f}°C | "
          f"val loss: {val_loss:.4f}, val MAE: {val_mae:.2f}°C")

    for cb in callbacks:
        cb.step(metrics, model)

    if any(isinstance(cb, EarlyStopping) and cb.should_stop for cb in callbacks):
        print(f"  Stopped at epoch {epoch}")
        break

writer.close()

Epoch 01 — train loss: 130.8822, train MAE: 9.49°C | val loss: 102.7822, val MAE: 8.27°C
  ✓ Best model saved (val_mae: 8.27°C)
Epoch 02 — train loss: 80.0434, train MAE: 7.19°C | val loss: 64.2908, val MAE: 6.41°C
  ✓ Best model saved (val_mae: 6.41°C)
Epoch 03 — train loss: 53.3876, train MAE: 5.76°C | val loss: 42.6147, val MAE: 5.10°C
  ✓ Best model saved (val_mae: 5.10°C)
Epoch 04 — train loss: 37.4387, train MAE: 4.71°C | val loss: 29.5318, val MAE: 4.13°C
  ✓ Best model saved (val_mae: 4.13°C)
Epoch 05 — train loss: 27.6335, train MAE: 3.97°C | val loss: 21.8678, val MAE: 3.49°C
  ✓ Best model saved (val_mae: 3.49°C)
Epoch 06 — train loss: 21.7849, train MAE: 3.49°C | val loss: 17.8419, val MAE: 3.15°C
  ✓ Best model saved (val_mae: 3.15°C)
Epoch 07 — train loss: 18.5473, train MAE: 3.23°C | val loss: 15.4830, val MAE: 2.96°C
  ✓ Best model saved (val_mae: 2.96°C)
Epoch 08 — train loss: 16.5461, train MAE: 3.07°C | val loss: 13.8684, val MAE: 2.83°C
  ✓ Best model saved (val_mae

Exception in thread Thread-32:
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/tensorboard/summary/writer/event_file_writer.py", line 244, in run
    self._run()
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/tensorboard/summary/writer/event_file_writer.py", line 275, in _run
    self._record_writer.write(data)
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/tensorboard/summary/writer/record_writer.py", line 40, in write
Exception in thread Thread-35:
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
Exception in thread Thread-33:
Traceback (most recent call last):
  File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/threading.py", line 1075, in _bootst

Epoch 15 — train loss: 10.7618, train MAE: 2.56°C | val loss: 9.6498, val MAE: 2.42°C
  ✓ Best model saved (val_mae: 2.42°C)


FileNotFoundError: [Errno 2] No such file or directory: b'runs/jena_bidirectional_rnn/Loss_train/events.out.tfevents.1773860954.cs-01km0np14zzm1z7jw1g1zj1ah2.369319.28'

In [ ]:
model.load_state_dict(torch.load("jena_bidirectional_rnn_best.pt", map_location=device))
_, test_mae = run_epoch(model, test_loader, criterion)
print(f"\nTest MAE: {test_mae:.2f}°C")